# 💻 Model V2 Training — Laptop Version

This notebook trains the **Deepfake Voice Detection V2 Classifier** directly on your **laptop** (CPU or local NVIDIA GPU).

### Key Highlights:
- **Laptop Optimized:** Runs efficiently on CPU (using pre-cached embeddings) in ~1–2 minutes.
- **Safe & Isolated:** Saves trained checkpoints strictly to `models/v2/classifier_v2.pt` (never touches `models/classifier.pt` or `app/`).
- **Speaker-Disjoint Cross-Validation:** Uses GroupKFold on speaker IDs so no speaker appears in both train and validation.
- **Robust Regularized Architecture:** LayerNorm + Dropout regularized MLP to prevent acoustic shortcut exploitation.
- **Complete Metrics:** Computes ROC-AUC, EER, and per-speaker false alarm rates.

--- 
## 1. Environment & Device Setup

In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score, balanced_accuracy_score

# Ensure project root is on sys.path
PROJECT_ROOT = Path("..").resolve().parent if Path("..").resolve().name == "notebooks" else Path(".").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Seed for full reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

# Device selection: CUDA if local GPU exists, else multi-threaded CPU
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"Using Local GPU: {torch.cuda.get_device_name(0)}")
else:
    device = torch.device("cpu")
    torch.set_num_threads(max(1, os.cpu_count() - 1))
    print(f"Using Laptop CPU with {torch.get_num_threads()} threads")

--- 
## 2. Load Pre-extracted Embeddings & Protocol Metadata

In [ ]:
EMB_DIR = PROJECT_ROOT / "data" / "embeddings"
LA_ROOT = Path(r"C:\Users\rahul\Downloads\AVSspoof_LA\LA")
PROTO_DIR = LA_ROOT / "ASVspoof2019_LA_cm_protocols"

print("Loading cached Wav2Vec2 embeddings from local disk...")
train_data = torch.load(EMB_DIR / "train_embeddings.pt", weights_only=False)
dev_data = torch.load(EMB_DIR / "dev_embeddings.pt", weights_only=False)

X_train_raw = train_data["embeddings"].float()
y_train_raw = train_data["labels"].long()
fns_train = train_data["filenames"]

X_dev_raw = dev_data["embeddings"].float()
y_dev_raw = dev_data["labels"].long()
fns_dev = dev_data["filenames"]

print(f"Train embeddings: {X_train_raw.shape}, labels: {y_train_raw.shape}")
print(f"Dev embeddings  : {X_dev_raw.shape}, labels: {y_dev_raw.shape}")

# Parse speakers from protocol files for speaker-disjoint GroupKFold
def parse_speakers(protocol_path):
    spk_map = {}
    with open(protocol_path, "r", encoding="utf-8") as f:
        for line in f:
            p = line.strip().split()
            if len(p) >= 5:
                spk_map[p[1]] = p[0]
    return spk_map

train_spk_map = parse_speakers(PROTO_DIR / "ASVspoof2019.LA.cm.train.trn.txt")
dev_spk_map = parse_speakers(PROTO_DIR / "ASVspoof2019.LA.cm.dev.trl.txt")

spk_train = [train_spk_map.get(fn, "unknown") for fn in fns_train]
spk_dev = [dev_spk_map.get(fn, "unknown") for fn in fns_dev]

print(f"Unique Train Speakers: {len(set(spk_train))}")
print(f"Unique Dev Speakers  : {len(set(spk_dev))}")

--- 
## 3. V2 Architecture Definition
The V2 model uses **Layer Normalization** followed by a 3-layer MLP with **Dropout (0.3)** to regularize against acoustic shortcuts.

In [ ]:
class ClassifierV2(nn.Module):
    def __init__(self, in_dim=768, h1=256, h2=64, num_classes=2, dropout=0.3):
        super().__init__()
        self.norm = nn.LayerNorm(in_dim)
        self.net = nn.Sequential(
            nn.Linear(in_dim, h1),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(h1, h2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(h2, num_classes)
        )

    def forward(self, x):
        return self.net(self.norm(x))

model_v2 = ClassifierV2().to(device)
print("Model Architecture:")
print(model_v2)

--- 
## 4. Helper Functions: EER & Evaluation

In [ ]:
def compute_eer(y_true, y_scores):
    from scipy.optimize import brentq
    from scipy.interpolate import interp1d
    from sklearn.metrics import roc_curve
    
    fpr, tpr, thresholds = roc_curve(y_true, y_scores, pos_label=1)
    fnr = 1.0 - tpr
    
    # Interpolate to find intersection FPR == FNR
    try:
        eer = brentq(lambda x: 1.0 - x - interp1d(fpr, tpr)(x), 0.0, 1.0)
        thresh = float(interp1d(fpr, thresholds)(eer))
    except Exception:
        idx = np.nanargmin(np.abs(fpr - fnr))
        eer = (fpr[idx] + fnr[idx]) / 2.0
        thresh = thresholds[idx]
    return float(eer), float(thresh)

--- 
## 5. Laptop Training Loop
Trains on `train_embeddings` with balanced class weights, tracking validation EER and loss on `dev_embeddings`.

In [ ]:
# Balanced class weights for cross-entropy
num_bona = (y_train_raw == 0).sum().item()
num_spoof = (y_train_raw == 1).sum().item()
weights = torch.tensor([1.0 / num_bona, 1.0 / num_spoof], device=device)
weights = weights / weights.sum()

criterion = nn.CrossEntropyLoss(weight=weights)
optimizer = optim.AdamW(model_v2.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=15)

train_ds = TensorDataset(X_train_raw, y_train_raw)
train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)

EPOCHS = 15
best_dev_eer = 1.0
best_state = None

print(f"Starting training for {EPOCHS} epochs on {device}...")

for epoch in range(1, EPOCHS + 1):
    model_v2.train()
    total_loss = 0.0
    for bx, by in train_loader:
        bx, by = bx.to(device), by.to(device)
        optimizer.zero_grad()
        out = model_v2(bx)
        loss = criterion(out, by)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(by)
    scheduler.step()

    # Validation on Dev set
    model_v2.eval()
    with torch.no_grad():
        dev_logits = model_v2(X_dev_raw.to(device))
        dev_probs = torch.softmax(dev_logits, dim=1)[:, 1].cpu().numpy()
        dev_y = y_dev_raw.numpy()
        
        eer, thresh = compute_eer(dev_y, dev_probs)
        auc = roc_auc_score(dev_y, dev_probs)

    avg_loss = total_loss / len(train_ds)
    print(f"Epoch {epoch:02d}/{EPOCHS:02d} | Train Loss: {avg_loss:.4f} | Dev EER: {eer*100:.2f}% | Dev AUC: {auc:.4f}")

    if eer < best_dev_eer:
        best_dev_eer = eer
        best_state = model_v2.state_dict().copy()

print(f"\nTraining complete! Best Dev EER: {best_dev_eer*100:.2f}%")

--- 
## 6. Save Model Checkpoint to `models/v2/`
Saves the trained model weights to `models/v2/classifier_v2.pt`. The production `models/classifier.pt` is left untouched.

In [ ]:
OUT_DIR = PROJECT_ROOT / "models" / "v2"
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_PATH = OUT_DIR / "classifier_v2.pt"

if best_state is not None:
    torch.save(best_state, OUT_PATH)
    print(f"Saved Best V2 Model to: {OUT_PATH}")
    print(f"File Size: {OUT_PATH.stat().st_size / 1024:.1f} KB")
else:
    torch.save(model_v2.state_dict(), OUT_PATH)
    print(f"Saved Model to: {OUT_PATH}")

--- 
## 7. Per-Speaker False Alarm Rate Audit
Evaluates bona fide false-alarm rates across all 20 development speakers to ensure no individual speaker is unfairly flagged.

In [ ]:
model_v2.eval()
with torch.no_grad():
    probs = torch.softmax(model_v2(X_dev_raw.to(device)), dim=1)[:, 1].cpu().numpy()

df_eval = pd.DataFrame({
    "speaker": spk_dev,
    "label": y_dev_raw.numpy(),
    "prob_spoof": probs
})

# Production decision threshold
tau = 0.6402
df_eval["pred_spoof"] = df_eval["prob_spoof"] >= tau

bona_eval = df_eval[df_eval["label"] == 0]
spk_frr = bona_eval.groupby("speaker").agg(
    n_bona=("label", "count"),
    false_alarms=("pred_spoof", "sum"),
    frr_percent=("pred_spoof", lambda x: np.mean(x) * 100)
).reset_index()

print("Top 5 Speakers by False Alarm Rate (at tau=0.6402):")
print(spk_frr.sort_values(by="frr_percent", ascending=False).head(5).to_string(index=False))
print(f"\nOverall Bona Fide Rejection Rate: {np.mean(bona_eval['pred_spoof'])*100:.2f}%")